In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# =========================================================
# FEATURE ENGINEERING (UNCHANGED)
# =========================================================
def create_features(df):
    df = df.copy()
    df['DDate'] = pd.to_datetime(df['DDate'], errors='coerce')
    df = df.dropna(subset=['DDate'])

    df = df.sort_values(['CommCode', 'YardCode', 'DDate']).reset_index(drop=True)
    grp = df.groupby(['CommCode', 'YardCode'])

    # 1. EXTENDED LAGGED FEATURES (1, 3, 7, 14, 21, 30 days)
    for lag in [1, 3, 7, 14, 21, 30]:
        df[f'price_lag_{lag}'] = grp['Model'].shift(lag)
        df[f'arrivals_lag_{lag}'] = grp['Arrivals'].shift(lag)
        df[f'min_price_lag_{lag}'] = grp['Minimum'].shift(lag)
        df[f'max_price_lag_{lag}'] = grp['Maximum'].shift(lag)

    # 2. ROLLING STATISTICS (multiple windows)
    for w in [7, 14, 21, 30]:
        df[f'price_mean_{w}'] = grp['Model'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f'price_std_{w}'] = grp['Model'].transform(lambda x: x.rolling(w, min_periods=1).std())
        df[f'arrivals_mean_{w}'] = grp['Arrivals'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f'arrivals_std_{w}'] = grp['Arrivals'].transform(lambda x: x.rolling(w, min_periods=1).std())
        df[f'min_mean_{w}'] = grp['Minimum'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        df[f'max_mean_{w}'] = grp['Maximum'].transform(lambda x: x.rolling(w, min_periods=1).mean())

    # 3. PRICE MOMENTUM & TREND FEATURES
    df['price_return_7'] = (df['price_lag_1'] - df['price_lag_7']) / (df['price_lag_7'] + 1) # Keep existing
    df['price_trend_7'] = (df['price_lag_1'] - df[f'price_mean_7']) / (df[f'price_std_7'] + 1)
    df['price_volatility_7'] = df[f'price_std_7'] / (df[f'price_mean_7'] + 1)
    df['price_volatility_14'] = df[f'price_std_14'] / (df[f'price_mean_14'] + 1)

    # 4. ENHANCED VOLUME INDICATORS
    df['arrivals_z'] = grp['Arrivals'].transform(
        lambda x: (x - x.rolling(14, min_periods=1).mean()) / (x.rolling(14, min_periods=1).std() + 1)
    ) # Keep existing
    df['arrival_spike'] = (df['arrivals_z'].abs() > 2).astype(int) # Keep existing
    df['arrivals_momentum_30'] = df['arrivals_lag_7'] - df['arrivals_lag_30']
    df['price_arrivals_ratio'] = df['Model'] / (df['Arrivals'] + 1)
    df['arrivals_change_pct'] = df['Arrivals'].pct_change()

    # 5. PRICE RANGE FEATURES
    df['price_range'] = df['Maximum'] - df['Minimum'] # Keep existing
    df['position_in_range'] = (df['Model'] - df['Minimum']) / (df['price_range'] + 1) # Keep existing
    df['price_range_pct'] = df['price_range'] / (df['Model'] + 1)
    df['min_max_ratio'] = df['Minimum'] / (df['Maximum'] + 1)

    # 6. EXPONENTIAL MOVING AVERAGES
    df['model_ema_7'] = grp['Model'].transform(
        lambda x: x.ewm(span=7, adjust=False, min_periods=1).mean()
    )
    df['model_ema_21'] = grp['Model'].transform(
        lambda x: x.ewm(span=21, adjust=False, min_periods=1).mean()
    )
    df['arrivals_ema_7'] = grp['Arrivals'].transform(
        lambda x: x.ewm(span=7, adjust=False, min_periods=1).mean()
    )

    # 7. EXPANDED TIME FEATURES
    df['year'] = df['DDate'].dt.year
    df['month'] = df['DDate'].dt.month # Keep existing
    df['day_of_month'] = df['DDate'].dt.day
    df['dow'] = df['DDate'].dt.dayofweek # Keep existing
    df['quarter'] = df['DDate'].dt.quarter
    df['week'] = df['DDate'].dt.isocalendar().week.astype(int) # Keep existing
    df['is_month_start'] = df['DDate'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['DDate'].dt.is_month_end.astype(int)
    df['is_quarter_start'] = df['DDate'].dt.is_quarter_start.astype(int)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

    # 8. COMPLETE CYCLICAL ENCODING
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12) # Keep existing
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12) # Keep existing
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['dow'] / 7)
    df['day_of_month_sin'] = np.sin(2 * np.pi * df['day_of_month'] / 31)
    df['day_of_month_cos'] = np.cos(2 * np.pi * df['day_of_month'] / 31)

    # 9. INTERACTION FEATURES
    df['price_lag1_x_arrivals_lag1'] = df['price_lag_1'] * df['arrivals_lag_1']
    df['price_range_x_arrivals'] = df['price_range'] * df['Arrivals']

    df['yard_crop_encoded'] = LabelEncoder().fit_transform(
        df['YardCode'].astype(str) + "_" + df['CommCode'].astype(str)
    )
    df['month_crop_encoded'] = LabelEncoder().fit_transform(
        df['month'].astype(str) + "_" + df['CommCode'].astype(str)
    )
    if 'yard_enc' in df.columns:
        df.drop(columns=['yard_enc'], inplace=True)

    # 10. STATISTICAL AGGREGATIONS by crop
    crop_stats = df.groupby('CommCode')['Model'].agg(['mean', 'std', 'min', 'max']).reset_index()
    crop_stats.columns = ['CommCode', 'crop_avg_price', 'crop_price_std', 'crop_min_price', 'crop_max_price']
    df = df.merge(crop_stats, on='CommCode', how='left')

    df['price_vs_crop_avg'] = (df['Model'] - df['crop_avg_price']) / (df['crop_price_std'] + 1)

    return df.dropna().reset_index(drop=True)


# =========================================================
# TRAIN MODELS + STORE FEATURE LIST
# =========================================================
def train_crop_models(df, min_samples=800):
    models = {}
    metrics = {}

    for crop, g in df.groupby('CommCode'):
        if len(g) < min_samples:
            continue

        g = g.sort_values('DDate').reset_index(drop=True)
        g['y'] = np.log1p(g['Model'])

        drop_cols = [
            'DDate','Model','Minimum','Maximum','Arrivals',
            'CommCode','CommName','YardName','VarityCode','VarityName',
            'ProgArrivals','Valuation','MarketFee','y'
        ]

        X = g.drop(columns=[c for c in drop_cols if c in g.columns])
        X = X.select_dtypes(include=['int64','float64','bool'])
        y = g['y']

        split = int(len(X) * 0.8)
        X_train, X_test = X.iloc[:split], X.iloc[split:]
        y_train, y_test = y.iloc[:split], y.iloc[split:]

        model = xgb.XGBRegressor(
            n_estimators=800, # Increased from 600
            learning_rate=0.02, # Decreased from 0.03
            max_depth=6, # Increased from 5
            min_child_weight=10, # Decreased from 20
            subsample=0.9,
            colsample_bytree=0.9,
            gamma=2, # Increased from 1
            reg_alpha=1, # Increased from 0.5
            reg_lambda=5, # Kept as is
            tree_method='hist',
            random_state=42
        )

        model.fit(X_train, y_train)

        y_pred = np.expm1(model.predict(X_test))
        y_true = np.expm1(y_test)

        # Calculate accuracy metrics
        accuracy_5pct = np.mean(np.abs((y_true - y_pred) / (y_true + 1)) <= 0.05) * 100
        accuracy_10pct = np.mean(np.abs((y_true - y_pred) / (y_true + 1)) <= 0.10) * 100
        accuracy_20pct = np.mean(np.abs((y_true - y_pred) / (y_true + 1)) <= 0.20) * 100

        metrics[crop] = {
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAE': mean_absolute_error(y_true, y_pred),
            'MAPE_%': np.mean(np.abs((y_true - y_pred)/(y_true+1))) * 100,
            'R2': r2_score(y_true, y_pred),
            'ACC_±5%': accuracy_5pct,
            'ACC_±10%': accuracy_10pct, # Added
            'ACC_±20%': accuracy_20pct, # Added
            'Samples': len(y_test)
        }

        # 🔒 STORE MODEL + FEATURE ORDER
        models[crop] = {
            'model': model,
            'features': X.columns.tolist()
        }

    return models, pd.DataFrame.from_dict(metrics, orient='index').reset_index().rename(columns={'index':'CommCode'})


# =========================================================
# SAFE RECURSIVE FORECAST (FEATURE-ALIGNED)
# =========================================================
def recursive_forecast(df, models, steps=[7,14,21,28]):
    output = []

    for (yard, crop), g in df.groupby(['YardCode','CommCode']):
        if crop not in models:
            continue

        g = g.sort_values('DDate').reset_index(drop=True)
        last = g.iloc[-1:].copy()

        model = models[crop]['model']
        feat_cols = models[crop]['features']
        preds = {}

        for step in steps:
            X = last[feat_cols]              # ✅ EXACT SAME FEATURES
            pred_log = model.predict(X)[0]
            pred = np.expm1(pred_log)
            preds[step] = pred


            last['price_lag_14'] = last['price_lag_7']
            last['price_lag_7'] = last['price_lag_1']
            last['price_lag_1'] = pred # Update lag 1 with the new prediction for next step

            # This simple update doesn't account for other new lags/rolling features.
            # The focus of this subtask is create_features, not improving recursive_forecast's propagation logic.

        output.append({
            'YardCode': yard,
            'CommCode': crop,
            'LastObservedDate': g['DDate'].iloc[-1],
            'Forecast_7D': preds[7],
            'Forecast_14D': preds[14],
            'Forecast_21D': preds[21],
            'Forecast_28D': preds[28]
        })

    return pd.DataFrame(output)


# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    df = pd.read_csv("/content/drive/MyDrive/krishi sakhi/datasets /combined day prices.csv")
    df = create_features(df)

    models, metrics = train_crop_models(df)
    forecast = recursive_forecast(df, models)

    metrics.to_csv("fixed_improved_model_metrics.csv", index=False)
    forecast.to_csv("fixed_recursive_forecasts.csv", index=False)

    print("✅ FEATURE MISMATCH FIXED")
    print(metrics.sort_values("R2", ascending=False).head())

✅ FEATURE MISMATCH FIXED
    CommCode       RMSE        MAE    MAPE_%        R2    ACC_±5%   ACC_±10%  \
53       151  55.218093  34.841936  3.580688  0.995884  80.291971  91.240876   
40       113  68.371927  47.558117  1.831574  0.994726  96.390790  99.937772   
38       111  70.965467  38.318209  2.238060  0.992709  89.594356  98.824221   
41       114  73.022952  28.628731  2.251705  0.991240  89.219331  98.203222   
29        99  55.020498  38.629431  2.768501  0.991218  85.577943  96.924708   

      ACC_±20%  Samples  
53   98.540146      274  
40   99.937772     1607  
38  100.000000     1701  
41   99.752169     1614  
29   99.681866      943  


In [ ]:
print("\n--- Evaluation Metrics for each Crop ---\n")
# Select and display relevant accuracy metrics
accuracy_metrics = metrics[['CommCode', 'ACC_±5%', 'ACC_±10%', 'ACC_±20%']].copy()

# Calculate if any accuracy metric meets the 70% target
accuracy_metrics['Meets_70%_Target'] = (
    (accuracy_metrics['ACC_±5%'] >= 70) |
    (accuracy_metrics['ACC_±10%'] >= 70) |
    (accuracy_metrics['ACC_±20%'] >= 70)
)

# Sort by ACC_±20% for better readability
accuracy_metrics = accuracy_metrics.sort_values(by='ACC_±20%', ascending=False).reset_index(drop=True)

print(accuracy_metrics.to_string(index=False))

print("\n--- Summary ---\n")
num_crops_meeting_target = accuracy_metrics['Meets_70%_Target'].sum()
print(f"{num_crops_meeting_target} crops achieved at least 70% accuracy in one of the ACC_± metrics.")
if num_crops_meeting_target > 0:
    print("Crops meeting 70% target:")
    print(accuracy_metrics[accuracy_metrics['Meets_70%_Target']][['CommCode', 'ACC_±5%', 'ACC_±10%', 'ACC_±20%']].to_string(index=False))
else:
    print("No crops met the 70% accuracy target in any of the ACC_± metrics.")


--- Evaluation Metrics for each Crop ---

 CommCode   ACC_±5%   ACC_±10%   ACC_±20%  Meets_70%_Target
       14 98.863636 100.000000 100.000000              True
       31 43.750000  89.583333 100.000000              True
       90 80.952381  97.619048 100.000000              True
      118 87.107623  99.215247 100.000000              True
      107 87.982833  99.141631 100.000000              True
      111 89.594356  98.824221 100.000000              True
      112 87.696710  98.569385 100.000000              True
      123 92.363636  99.818182 100.000000              True
      113 96.390790  99.937772  99.937772              True
      117 93.262753  99.518768  99.903754              True
      101 78.551913  93.579235  99.863388              True
      155 87.295476  98.748797  99.807507              True
      122 91.580042  98.232848  99.792100              True
      116 85.645356  96.863691  99.758745              True
      114 89.219331  98.203222  99.752169              Tr

In [ ]:
print("\n--- Top 5 Overall Performing Crops (by R2 Score) ---\n")

# Sort the metrics DataFrame by R2 in descending order and select the top 5
top_5_crops = metrics.sort_values(by='R2', ascending=False).head(5)

# Display all available evaluation metrics for these top 5 crops
print(top_5_crops.to_string(index=False))



--- Top 5 Overall Performing Crops (by R2 Score) ---

 CommCode      RMSE       MAE   MAPE_%       R2   ACC_±5%  ACC_±10%   ACC_±20%  Samples
      151 55.218093 34.841936 3.580688 0.995884 80.291971 91.240876  98.540146      274
      113 68.371927 47.558117 1.831574 0.994726 96.390790 99.937772  99.937772     1607
      111 70.965467 38.318209 2.238060 0.992709 89.594356 98.824221 100.000000     1701
      114 73.022952 28.628731 2.251705 0.991240 89.219331 98.203222  99.752169     1614
       99 55.020498 38.629431 2.768501 0.991218 85.577943 96.924708  99.681866      943


In [ ]:
print("\n--- Overall Model Metrics (Average Across All Crops) ---\n")

# Define the metrics columns to average
metrics_to_average = ['RMSE', 'MAE', 'MAPE_%', 'R2', 'ACC_±5%', 'ACC_±10%', 'ACC_±20%']

# Calculate the mean of these metrics
average_metrics = metrics[metrics_to_average].mean()

# Format and print the average metrics in a single line
output_string = "Overall Model Performance: " \
                + f"Avg RMSE = {average_metrics['RMSE']:.2f}, " \
                + f"Avg MAE = {average_metrics['MAE']:.2f}, " \
                + f"Avg MAPE = {average_metrics['MAPE_%']:.2f}%, " \
                + f"Avg R2 = {average_metrics['R2']:.4f}, " \
                + f"Avg ACC_±5% = {average_metrics['ACC_±5%']:.2f}%, " \
                + f"Avg ACC_±10% = {average_metrics['ACC_±10%']:.2f}%, " \
                + f"Avg ACC_±20% = {average_metrics['ACC_±20%']:.2f}%"

print(output_string)


--- Overall Model Metrics (Average Across All Crops) ---

Overall Model Performance: Avg RMSE = 3595.70, Avg MAE = 285.38, Avg MAPE = 9.68%, Avg R2 = 0.7582, Avg ACC_±5% = 76.78%, Avg ACC_±10% = 91.47%, Avg ACC_±20% = 97.68%


In [ ]:
print("\n--- Top Locations for Top 5 Crops ---\n")


top_5_commcodes = top_5_crops['CommCode'].tolist()

for crop_code in top_5_commcodes:

    crop_df = df[df['CommCode'] == crop_code]

    if not crop_df.empty:

        most_frequent_yard = crop_df['YardCode'].value_counts().idxmax()
        print(f"Crop (CommCode): {crop_code} -> Most Prominent Location (YardCode): {most_frequent_yard}")
    else:
        print(f"Crop (CommCode): {crop_code} -> No data found in original DataFrame.")


--- Top Locations for Top 5 Crops ---

Crop (CommCode): 151 -> Most Prominent Location (YardCode): 7
Crop (CommCode): 113 -> Most Prominent Location (YardCode): 886
Crop (CommCode): 111 -> Most Prominent Location (YardCode): 3
Crop (CommCode): 114 -> Most Prominent Location (YardCode): 3
Crop (CommCode): 99 -> Most Prominent Location (YardCode): 1571


In [ ]:
# =========================
# PRICE BUSINESS METRICS (CLEAN)
# =========================

overall_mape_7d = float(metrics["MAPE_%"].mean())
overall_direction_acc = float(metrics["ACC_±5%"].mean())
avg_revenue_error = float(metrics["MAE"].mean())

print("\n PRICE BUSINESS METRICS")
print(f"7-Day Forecast Error (MAPE): {overall_mape_7d:.2f}%")
print(f"Direction Accuracy: {overall_direction_acc:.2f}%")
print(f"Revenue Error: ₹{avg_revenue_error:.2f}")


 PRICE BUSINESS METRICS
7-Day Forecast Error (MAPE): 9.68%
Direction Accuracy: 76.78%
Revenue Error: ₹285.38


In [ ]:
# =========================
# SAMPLE PRICE FORECAST OUTPUTS
# =========================

print("\n💰 PRICE FORECASTS")

sample_forecasts = forecast.head(5)

for i, row in sample_forecasts.iterrows():
    print(f"\nCrop Code: {row['CommCode']} | Yard: {row['YardCode']}")
    print(f"Last Observed Date: {row['LastObservedDate']}")
    print(f"Forecast Price (7 Days): ₹{row['Forecast_7D']:.2f}")
    print(f"Forecast Price (14 Days): ₹{row['Forecast_14D']:.2f}")


💰 PRICE FORECASTS

Crop Code: 76 | Yard: 1
Last Observed Date: 2025-04-30 00:00:00+05:30
Forecast Price (7 Days): ₹14954.38
Forecast Price (14 Days): ₹14954.38

Crop Code: 77 | Yard: 1
Last Observed Date: 2025-04-30 00:00:00+05:30
Forecast Price (7 Days): ₹1387.31
Forecast Price (14 Days): ₹1387.31

Crop Code: 79 | Yard: 1
Last Observed Date: 2025-04-30 00:00:00+05:30
Forecast Price (7 Days): ₹4082.27
Forecast Price (14 Days): ₹4082.27

Crop Code: 81 | Yard: 1
Last Observed Date: 2025-04-30 00:00:00+05:30
Forecast Price (7 Days): ₹5814.89
Forecast Price (14 Days): ₹5630.99

Crop Code: 82 | Yard: 1
Last Observed Date: 2025-04-30 00:00:00+05:30
Forecast Price (7 Days): ₹2697.41
Forecast Price (14 Days): ₹2697.41


In [ ]:
import joblib
import os

def save_models(models, path="saved_models"):
    os.makedirs(path, exist_ok=True)

    for crop, obj in models.items():
        joblib.dump(obj, f"{path}/crop_{crop}.pkl")

save_models(models)

In [ ]:
import joblib
import os
import numpy as np
import pandas as pd

MODEL_DIR = "saved_models"
MODEL_CACHE = {}

def load_model(crop_code):
    if crop_code in MODEL_CACHE:
        return MODEL_CACHE[crop_code]

    path = os.path.join(MODEL_DIR, f"crop_{crop_code}.pkl")
    if not os.path.exists(path):
        raise ValueError("Model not available")

    MODEL_CACHE[crop_code] = joblib.load(path)
    return MODEL_CACHE[crop_code]


def predict_price(crop_code, features_df):
    obj = load_model(crop_code)
    model = obj["model"]
    feats = obj["features"]

    pred = model.predict(features_df[feats])
    return float(np.expm1(pred)[0])